In [12]:
#!pip install  accelerate bitsandbytes -q


In [13]:
#pip install --upgrade transformers


In [14]:
#!pip install -q peft 

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset


2025-06-20 05:56:28.250023: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750398988.501023      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750398988.572235      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
df = pd.read_csv("/kaggle/input/ai-medical-chatbot/ai-medical-chatbot.csv")


In [3]:
df.head()

,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256916 entries, 0 to 256915
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Description  256916 non-null  object
 1   Patient      256916 non-null  object
 2   Doctor       256916 non-null  object
dtypes: object(3)
memory usage: 5.9+ MB


In [19]:
df.duplicated().sum()

10378

Nous avons 10378 duplicated values sur 256916 entrees, donc on peu les drop

#### Netoyage des donnees

In [4]:
# Supprimer les lignes nulles ou doublons
df.dropna(subset=['Description', 'Patient', 'Doctor'], inplace=True)
df.drop_duplicates(inplace=True)


 #### Normalisation des Textes

In [5]:
import re

def clean_text(text):
    text = text.lower()                             # Minuscule
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # Supprimer les URLs
    text = re.sub(r'\@\w+|\#','', text)             # Supprimer mentions et hashtags
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)      # Supprimer ponctuation
    text = re.sub(r'\s+', ' ', text).strip()        # Supprimer espaces superflus
    return text

df['Description'] = df['Description'].apply(clean_text)
df['Patient'] = df['Patient'].apply(clean_text)
df['Doctor'] = df['Doctor'].apply(clean_text)


#### Tokenisation

#### Suppression des Stopwords

In [22]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

df['tokens_patient'] = df['tokens_patient'].apply(lambda x: [word for word in x if word not in stop_words])
df['tokens_doctor'] = df['tokens_doctor'].apply(lambda x: [word for word in x if word not in stop_words])

NameError: name 'nltk' is not defined

#### Vu que nos phrases doivent garder le contexte, on va utiliser Lemmatization 

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
df['tokens_patient'] = df['tokens_patient'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])


In [6]:
# Créer une colonne "text" avec le format d'entraînement
df["text"] = "Patient: " + df["Patient"] + "\nDoctor: " + df["Doctor"]

In [7]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2)
val_df, test_df = train_test_split(val_df,test_size=0.1)

train_dataset = Dataset.from_pandas(train_df[["text"]])
val_dataset = Dataset.from_pandas(val_df[["text"]])
test_dataset = Dataset.from_pandas(test_df[["text"]])


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType

model_name = "tiiuae/falcon-rw-1b"

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map={"": torch.cuda.current_device()},
    quantization_config=bnb_config
)

model.config.pad_token_id = tokenizer.pad_token_id


tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [9]:
print(model.hf_device_map)


NameError: name 'model' is not defined

In [72]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_key_value"],  # Pour Falcon
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 1,572,864 || all params: 1,313,198,080 || trainable%: 0.1198


#### Tokenization

In [73]:
from datasets import Dataset

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_dataset.map(tokenize)
tokenized_val = val_dataset.map(tokenize)


Map:   0%|          | 0/197230 [00:00<?, ? examples/s]

Map:   0%|          | 0/44377 [00:00<?, ? examples/s]

Arguments d'entrainement

In [74]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./falcon-medbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=0.0020,
    logging_dir='./logs',
    logging_steps=10,
    label_names=["input_ids"],
    save_total_limit=2,
    save_strategy="epoch",
    eval_strategy="steps",  
    fp16=True,                 # si GPU compatible
    report_to="none"
)


#####  Entraîneur

In [75]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,  
    data_collator=data_collator,
    
)

In [76]:
import torch
torch.cuda.empty_cache()

!nvidia-smi  # pour vérifier l’usage GPU sur Kaggle


Fri Jun 20 03:37:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             31W /   70W |    5093MiB /  15360MiB |     88%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss
10,3.300500,3.242041


In [ ]:
model.save_pretrained("falcon-lora-finetuned")
tokenizer.save_pretrained("falcon-lora-finetuned")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# Charger le tokenizer
tokenizer = AutoTokenizer.from_pretrained("falcon-lora-finetuned")
tokenizer.pad_token = tokenizer.eos_token  # si nécessaire

# Charger le modèle LoRA fine-tuné
base_model = AutoModelForCausalLM.from_pretrained("tiiuae/falcon-rw-1b", device_map="auto")
model = PeftModel.from_pretrained(base_model, "falcon-lora-finetuned")
model.eval()


In [ ]:
prompt = "### Question : I feel chest pain when I run, what should I do?\n### Réponse :"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
